In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd

import psutil
import math
import gc

import os, random
import posixpath
from pyhdas.frequency import spectrogram, add_db, energy
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta

In [2]:
# select 200 random files from the directory
normal_data_dir = "data/19"
files = random.sample(os.listdir(normal_data_dir), 200)

In [ ]:
for file in files:

    filepath = [posixpath.join(normal_data_dir, file)]

    # open file one by one
    ds_raw = concat_raw_data(filepath)

    # select locations 4210-4260
    poi = np.arange(4210, 4270, 10)

    ds_raw = ds_raw.sel(position=poi)
    ds_spect = spectrogram(ds_raw, variable='strain')
    
    n_positions = len(poi)
    max_columns = 2
    n_rows = math.ceil(n_positions / max_columns)
    fig, axes = plt.subplots(n_rows, max_columns, figsize=(15, 5 * n_rows), constrained_layout=True)
    axes = axes.flatten()

    # for every location, make a spectrogram plot
    for idx, pos in enumerate(ds_spect.position):
        ax = axes[idx] 
        ds_spect.Pxx_dB.sel(position=pos).plot(
            x="time", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma"
        )
        ax.set_title(f"Spectrogram at {pos.values:.1f}m")
        ax.set_xlabel("Time")
        ax.set_ylabel("Frequency [Hz]")

    for ax in axes[n_positions:]:
        fig.delaxes(ax)
    
    # save in the normal data directory
    output_folder = Path(f"normal_spectrogram_plots")
    output_folder.mkdir(parents=True, exist_ok=True)

    plot_filename = f"{file[:-4]}_spectrogram.png"
    plt.savefig(output_folder / plot_filename)

    plt.close(fig)
    
    print(f"Done with {file}")
    del ds_raw, ds_spect
    gc.collect()
    break

trigger_frequency: 2000.0
spatial_sampling: 10.0
fiber_position_offset: 60.0
das_type: aragon
FileHeaderSize: 200.0
Global_TimeProcessing_Spatial_Sampling_(Meters): 10.0
Global_TimeProcessing_Spatial_Sampling_(Points): 100.0
Global_TraceStats_Fiber_Length_Monitored_(Meters): 8000.0
Global_TraceStats_Fiber_Length_Monitored_(Points): 80000.0
Global_DigitizerData_Trace_SampleRate (GS_s): 1.0
Global_RAM_User_SET_Trigger_Freq: 2000.0
Global_TimeProcessing_Min._Trace_Block_(MTB): 40.0
Global_RAM_User_SET_Pulse_Width_(meter): 10.0
Global_RAM_User_SET_Sensitivity_(Chirp_Slope): 0.0
Global_DigitizerData_Correlation_Window_Size_(points): 100.0
Global_TimeProcessing_Strain_Fiber_Processed_Start_Point_(meters): 60.0
Global_TimeProcessing_Strain_Fiber Processed_Start_Point_(Integer): 6.0
Global_TimeProcessing_Strain_Fiber_Processed_Stop_Point_(Meter): 7100.0
Global_TimeProcessing_Strain_Fiber_Processed_Stop_Point_(Integer): 710.0
Global_TimeProcessing_Number_of_Averages_(Dynamic Processing, Strain)

In [ ]:
# open the event table
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])
# events = events[(events["start"] > pd.Timestamp("2021-02-26 07:25:44").tz_localize(None))]

In [ ]:
# loop through every event
for _, event in events.iterrows():
        
        # extract start and end times and locations
        start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
        poi = np.array([int(x.strip()) for x in poi.split(",")])

        # if the duration of the event is more than 2 minutes, print the event_label, date and start and end time, and move on to the next row
        event_duration = (end - start).total_seconds()
        if event_duration > 120 or event_duration <= 1:
            continue
        
        # load the strain data based on the start and the end of the event
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)
        start = start.tz_localize("UTC")
        end = end.tz_localize("UTC")
        day = start.day

        dir_data = Path(fr"data/{day}")
        file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        ds_raw = concat_raw_data(file_list)

        start = start.tz_localize(None)
        end = end.tz_localize(None)
        ds_raw = ds_raw.sel(time=slice(start, end), position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain')

        # Create a figure for all positions
        # 1 figure will contain N/2 rows with 2 column (depending on the number of locations) of spectograms
        n_positions = len(poi)
        max_columns = 2
        n_rows = math.ceil(n_positions / max_columns)
        fig, axes = plt.subplots(n_rows, max_columns, figsize=(15, 5 * n_rows), constrained_layout=True)

        axes = axes.flatten()
        flag = True
        
        if n_positions == 1:  
            fig, axes = plt.subplots(n_positions, 1, figsize=(10, 5 * n_positions), constrained_layout=True)
            axes = [axes]
            flag = False

        # for every location, make a spectrogram plot
        for idx, pos in enumerate(ds_spect.position):
            ax = axes[idx] 
            ds_spect.Pxx_dB.sel(position=pos).plot(
                x="time", y="freq", ax=ax, vmin=-40, vmax=40, cmap="magma"
            )
            ax.set_title(f"Spectrogram at {pos.values:.1f}m")
            ax.set_xlabel("Time")
            ax.set_ylabel("Frequency [Hz]")

        if flag:
            for ax in axes[n_positions:]:
                fig.delaxes(ax)

        # save the plot in the folder of the corresponding label (if it does not exist yet, create this folder; the main data folder is called spectrogram_plots)
        output_folder = Path(f"spectrogram_plots/{label}")
        output_folder.mkdir(parents=True, exist_ok=True)
  
        plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_spectrogram.png"
        plt.savefig(output_folder / plot_filename)

        plt.close(fig)
        
        print(f"Done with {start}, {end}, {label}")
        del ds_raw, ds_spect
        gc.collect()

In [ ]:
def compute_spectral_centroid(spectrogram_data):
    
    freq = spectrogram_data['freq'].values 
    Pxx = spectrogram_data['Pxx'].values  

    # Calculate spectral centroid for each time step
    numerator = np.sum(Pxx * freq[:, np.newaxis], axis=0)  
    denominator = np.sum(Pxx, axis=0)  
    spectral_centroid = numerator / denominator  

    return spectral_centroid

In [ ]:
def compute_spectral_bandwidth(spectrogram_data):
    
    freq = spectrogram_data['freq'].values  # Frequencies (Hz)
    Pxx = spectrogram_data['Pxx'].values   # Power spectral density

    spectral_centroid = compute_spectral_centroid(spectrogram_data)

    # Calculate Spectral Bandwidth
    numerator_bandwidth = np.sum(Pxx * ((freq[:, np.newaxis] - spectral_centroid)**2), axis=0)
    denominator_bandwidth = np.sum(Pxx, axis=0)
    spectral_bandwidth = np.sqrt(numerator_bandwidth / denominator_bandwidth)

    return spectral_bandwidth

In [ ]:
def compute_spectral_flatness(spectrogram_data):

    Pxx = spectrogram_data['Pxx'].values  # Power spectral density
    
    geometric_mean = np.exp(np.mean(np.log(Pxx), axis=0))  # Geometric mean
    arithmetic_mean = np.mean(Pxx, axis=0)  # Arithmetic mean
    
    spectral_flatness = geometric_mean / arithmetic_mean
    return spectral_flatness

In [ ]:
for _, event in events.iterrows():
    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
    poi = np.array([int(x.strip()) for x in poi.split(",")])
    
    event_duration = (end - start).total_seconds()
    if event_duration > 120 or event_duration <= 1:
        continue

    start = pd.Timestamp(start).tz_localize("UTC")
    end = pd.Timestamp(end).tz_localize("UTC")
    dir_data = Path(fr"data/{start.day}")
    file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
    if not file_list:
        print(f"The list is empty: {start}, {end}, {label}")
        continue

    ds_raw = concat_raw_data(file_list)
    ds_raw = ds_raw.sel(time=slice(start.tz_localize(None), end.tz_localize(None)), position=poi)
    ds_spect = spectrogram(ds_raw, variable='strain')

    n_positions = len(poi)

    max_columns = 2
    n_rows = math.ceil(n_positions / max_columns)
    fig, axes = plt.subplots(n_rows, max_columns, figsize=(15, 5 * n_rows), constrained_layout=True)

    axes = axes.flatten()
    flag = True
    
    if n_positions == 1:  
        fig, axes = plt.subplots(n_positions, 1, figsize=(10, 5 * n_positions), constrained_layout=True)
        axes = [axes]
        flag = False

    # For every position, calculate and plot spectral centroid and bandwidth
    for idx, pos in enumerate(ds_spect.position):
        ax = axes[idx]
        spectral_centroid = compute_spectral_centroid(ds_spect.sel(position=pos))
        spectral_bandwidth = compute_spectral_bandwidth(ds_spect.sel(position=pos), spectral_centroid)

        ax.plot(ds_spect.time, spectral_centroid, label="Spectral Centroid", color='r')
        ax.plot(ds_spect.time, spectral_bandwidth, label="Spectral Bandwidth", color='b')
        ax.set_title(f"Spectral Features at {pos.values:.1f}m")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
        ax.grid(alpha=0.3)
        ax.legend(loc="upper right")

    if flag:
        for ax in axes[n_positions:]:
            fig.delaxes(ax)

    output_folder = Path(f"spectral_features_plots/{label}")
    output_folder.mkdir(parents=True, exist_ok=True)

    plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_bdw_and_sc.png"
    plt.savefig(output_folder / plot_filename)

    plt.close(fig)
    
    print(f"Done with {start}, {end}, {label}")
    del ds_raw, ds_spect
    gc.collect()

In [ ]:
for _, event in events.iterrows():
    # Extract start and end times and locations
    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]
    poi = np.array([int(x.strip()) for x in poi.split(",")])

    event_duration = (end - start).total_seconds()
    if event_duration > 120 or event_duration <= 1:
        continue

    start = pd.Timestamp(start).tz_localize("UTC")
    end = pd.Timestamp(end).tz_localize("UTC")
    day = start.day

    dir_data = Path(fr"data/{day}")
    file_list = list(aragon_select_files(dir_data, start, end, extension="bin"))
    if not file_list:
        print(f"The list is empty: {start}, {end}, {label}")
        continue

    ds_raw = concat_raw_data(file_list)
    ds_raw = ds_raw.sel(time=slice(start.tz_localize(None), end.tz_localize(None)), position=poi)
    ds_spect = spectrogram(ds_raw, variable='strain')

    # Create a figure for all positions
    n_positions = len(poi)
    max_columns = 2
    n_rows = math.ceil(n_positions / max_columns)
    fig, axes = plt.subplots(n_rows, max_columns, figsize=(15, 5 * n_rows), constrained_layout=True)

    axes = axes.flatten()
    flag = True
    
    if n_positions == 1:  
        fig, axes = plt.subplots(n_positions, 1, figsize=(10, 5 * n_positions), constrained_layout=True)
        axes = [axes]
        flag = False

    # For every position, calculate and plot spectral flatness
    for idx, pos in enumerate(ds_spect.position):
        ax = axes[idx]
        spectral_flatness = compute_spectral_flatness(ds_spect.sel(position=pos))
        time = ds_spect.time.values

        ax.plot(time, spectral_flatness, label="Spectral Flatness", color='g')
        ax.set_title(f"Spectral Flatness at {pos.values:.1f}m")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Spectral Flatness")
        ax.set_ylim(0,1)
        ax.legend()
        ax.grid(alpha=0.3)

    if flag:
        for ax in axes[n_positions:]:
            fig.delaxes(ax)

    # Save the plot
    output_folder = Path(f"spectral_flatness_plots/{label}")
    output_folder.mkdir(parents=True, exist_ok=True)
    plot_filename = f"{start.strftime('%Y-%m-%d_%H:%M:%S')}_{end.strftime('%H:%M:%S')}_spectral_flatness.png"
    plt.savefig(output_folder / plot_filename)
    plt.close(fig)

    print(f"Done with {start}, {end}, {label}")
    del ds_raw, ds_spect
    gc.collect()